# Actividad Extra: Exploracion del Catalogo Netflix

## Objetivos
- Explorar y limpiar un dataset de contenido de Netflix
- Analizar la distribucion de peliculas vs series
- Identificar paises productores y generos mas comunes
- Analizar la evolucion temporal del catalogo
- Trabajar con columnas de texto usando split y explode

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import pandas as pd

spark = SparkSession.builder \
    .appName("actividad_netflix") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

---
## Ejercicio 1: Lectura y exploracion del catalogo

Lee el archivo `netflix_titles.csv`, explora el schema, cuenta los registros y analiza los valores nulos por columna.

**Columnas:** show_id, type, title, director, cast, country, date_added, release_year, rating, duration, listed_in, description

In [ ]:
# =============================================================
# EJERCICIO 1: Lectura y exploracion
# =============================================================

# Leer el CSV
df = spark.read.csv("/home/jovyan/datos/netflix_titles.csv", header=True, inferSchema=True)

# Schema
df.printSchema()

# Primeras filas
df.show(5, truncate=False)

# Total de registros
print(f"Total de registros: {df.count()}")

# Conteo de nulls por columna
print("\nNulls por columna:")
df.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]
).show(truncate=False)

> **Nota docente:** El patron para contar nulls por columna usa una list comprehension de Python dentro de `select()`. `F.when(F.col(c).isNull(), c)` retorna el valor solo cuando es null, y `F.count()` cuenta los no-nulos del resultado (que son los nulls originales). Las columnas `director`, `cast`, `country` y `date_added` suelen tener muchos nulls. Es importante que los alumnos entiendan que los nulls no son errores sino datos faltantes que deben manejarse segun el contexto del analisis.

---
## Ejercicio 2: Distribucion peliculas vs series

Cuenta cuantas peliculas ("Movie") y series ("TV Show") hay en el catalogo. Crea un grafico de barras.

In [ ]:
# =============================================================
# EJERCICIO 2: Distribucion peliculas vs series
# =============================================================

# Contar por tipo
df_tipo = df.groupBy("type").count().orderBy("count", ascending=False)
df_tipo.show()

# Grafico de barras
pdf_tipo = df_tipo.toPandas()

plt.figure(figsize=(8, 5))
plt.bar(pdf_tipo["type"], pdf_tipo["count"], color=["#E50914", "#564D4A"])
plt.title("Distribucion de Contenido: Peliculas vs Series", fontsize=14)
plt.xlabel("Tipo")
plt.ylabel("Cantidad")

# Agregar etiquetas de valor sobre las barras
for i, (tipo, cantidad) in enumerate(zip(pdf_tipo["type"], pdf_tipo["count"])):
    plt.text(i, cantidad + 30, str(cantidad), ha="center", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

> **Nota docente:** El catalogo de Netflix tiene tipicamente mas peliculas que series. El color #E50914 es el rojo corporativo de Netflix, un detalle que los alumnos pueden apreciar. Las etiquetas de valor sobre las barras (`plt.text()`) mejoran la legibilidad del grafico. Error comun: los alumnos pueden intentar usar `value_counts()` de Pandas directamente; es mejor practicar el patron de Spark `groupBy().count()` y luego convertir para graficar.

---
## Ejercicio 3: Top 10 paises productores

La columna `country` puede contener multiples paises separados por coma. Divide la columna, cuenta cada pais individualmente y muestra el top 10 con un grafico de barras horizontales.

In [ ]:
# =============================================================
# EJERCICIO 3: Top 10 paises productores
# =============================================================

# Filtrar nulls, split y explode
df_paises = df.filter(F.col("country").isNotNull()) \
    .withColumn("pais", F.explode(F.split("country", ", "))) \
    .withColumn("pais", F.trim("pais"))

# Contar por pais y top 10
df_top_paises = df_paises.groupBy("pais").count() \
    .orderBy("count", ascending=False) \
    .limit(10)

df_top_paises.show(truncate=False)

# Grafico de barras horizontales
pdf_paises = df_top_paises.toPandas()

plt.figure(figsize=(10, 6))
plt.barh(pdf_paises["pais"], pdf_paises["count"], color="#E50914")
plt.title("Top 10 Paises Productores de Contenido Netflix", fontsize=14)
plt.xlabel("Cantidad de Titulos")
plt.ylabel("Pais")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

> **Nota docente:** El patron `split() + explode()` es fundamental para trabajar con columnas que contienen listas de valores en un solo string. `F.split("country", ", ")` convierte el string en un array, y `F.explode()` crea una fila por cada elemento del array. Esto significa que una pelicula coproducida por 3 paises generara 3 filas. `F.trim()` elimina espacios en blanco que pueden quedar despues del split. Estados Unidos domina ampliamente el catalogo. Los alumnos avanzados pueden explorar la diferencia entre `explode()` (ignora nulls) y `explode_outer()` (preserva nulls como filas con null).

---
## Ejercicio 4: Evolucion temporal del catalogo

Parsea la columna `date_added`, extrae el anio y cuenta cuanto contenido se agrego por anio. Crea un grafico de linea.

In [ ]:
# =============================================================
# EJERCICIO 4: Evolucion temporal
# =============================================================

# Filtrar nulls y parsear fecha
df_temporal = df.filter(F.col("date_added").isNotNull()) \
    .withColumn("fecha_added", F.to_date(F.trim("date_added"), "MMMM d, yyyy")) \
    .withColumn("anio_added", F.year("fecha_added"))

# Contar por anio
df_por_anio = df_temporal.groupBy("anio_added").count() \
    .orderBy("anio_added") \
    .filter(F.col("anio_added").isNotNull())

df_por_anio.show()

# Grafico de linea
pdf_anio = df_por_anio.toPandas()

plt.figure(figsize=(10, 5))
plt.plot(pdf_anio["anio_added"], pdf_anio["count"], marker="o", linewidth=2, color="#E50914")
plt.fill_between(pdf_anio["anio_added"], pdf_anio["count"], alpha=0.1, color="#E50914")
plt.title("Contenido Agregado a Netflix por Anio", fontsize=14)
plt.xlabel("Anio")
plt.ylabel("Cantidad de Titulos")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

> **Nota docente:** El formato de fecha "MMMM d, yyyy" corresponde a fechas como "September 25, 2021". Es importante usar `F.trim()` antes del parseo porque la columna `date_added` puede tener espacios al inicio. Si el parseo falla, los valores quedan como null. El grafico muestra el crecimiento acelerado del catalogo de Netflix, con un pico alrededor de 2019-2020. `fill_between()` agrega un area sombreada bajo la curva para mejor visualizacion. Los alumnos pueden separar el analisis por tipo (Movie vs TV Show) para ver tendencias diferenciadas.

---
## Ejercicio 5: Generos mas comunes

La columna `listed_in` contiene uno o mas generos separados por coma. Divide, cuenta cada genero individualmente y muestra el top 15.

In [ ]:
# =============================================================
# EJERCICIO 5: Generos mas comunes
# =============================================================

# Split y explode de generos
df_generos = df.filter(F.col("listed_in").isNotNull()) \
    .withColumn("genero", F.explode(F.split("listed_in", ", "))) \
    .withColumn("genero", F.trim("genero"))

# Top 15 generos
df_top_generos = df_generos.groupBy("genero").count() \
    .orderBy("count", ascending=False) \
    .limit(15)

df_top_generos.show(truncate=False)

# Grafico de barras horizontales
pdf_generos = df_top_generos.toPandas()

plt.figure(figsize=(10, 7))
plt.barh(pdf_generos["genero"], pdf_generos["count"], color="#E50914")
plt.title("Top 15 Generos en Netflix", fontsize=14)
plt.xlabel("Cantidad de Titulos")
plt.ylabel("Genero")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

> **Nota docente:** Este ejercicio refuerza el patron `split() + explode()` visto en el ejercicio 3. Los generos en Netflix son bastante granulares (ej: "International Movies", "Dramas", "Comedies"). Un titulo puede pertenecer a multiples generos, por lo que la suma total de conteos por genero sera mayor que el numero total de titulos. Los alumnos pueden extender este analisis cruzando generos con tipo (Movie vs TV Show) o con pais para ver preferencias regionales.

---
## Ejercicio 6: Analisis de duracion

Para **peliculas**, extrae los minutos de la columna `duration` (formato: "90 min"). Para **series**, extrae las temporadas (formato: "2 Seasons" o "1 Season"). Crea un histograma para cada tipo.

In [ ]:
# =============================================================
# EJERCICIO 6: Analisis de duracion
# =============================================================

# --- Peliculas: duracion en minutos ---
df_movies = df.filter(F.col("type") == "Movie") \
    .filter(F.col("duration").isNotNull()) \
    .withColumn("minutos", F.regexp_extract("duration", r"(\d+)", 1).cast("int")) \
    .filter(F.col("minutos") > 0)

print("Estadisticas de duracion de peliculas (minutos):")
df_movies.select("minutos").describe().show()

# Histograma de peliculas
pdf_movies = df_movies.select("minutos").toPandas()

plt.figure(figsize=(10, 5))
plt.hist(pdf_movies["minutos"], bins=30, color="#E50914", edgecolor="white", alpha=0.8)
plt.title("Distribucion de Duracion de Peliculas", fontsize=14)
plt.xlabel("Duracion (minutos)")
plt.ylabel("Frecuencia")
plt.axvline(pdf_movies["minutos"].mean(), color="black", linestyle="--", label=f"Media: {pdf_movies['minutos'].mean():.0f} min")
plt.legend()
plt.tight_layout()
plt.show()

# --- Series: numero de temporadas ---
df_shows = df.filter(F.col("type") == "TV Show") \
    .filter(F.col("duration").isNotNull()) \
    .withColumn("temporadas", F.regexp_extract("duration", r"(\d+)", 1).cast("int")) \
    .filter(F.col("temporadas") > 0)

print("Estadisticas de temporadas de series:")
df_shows.select("temporadas").describe().show()

# Grafico de barras de temporadas
df_temp_count = df_shows.groupBy("temporadas").count().orderBy("temporadas")
pdf_temp = df_temp_count.toPandas()

plt.figure(figsize=(10, 5))
plt.bar(pdf_temp["temporadas"].astype(str), pdf_temp["count"], color="#564D4A")
plt.title("Distribucion de Numero de Temporadas", fontsize=14)
plt.xlabel("Numero de Temporadas")
plt.ylabel("Cantidad de Series")
plt.tight_layout()
plt.show()

> **Nota docente:** `F.regexp_extract("duration", r"(\d+)", 1)` extrae el primer grupo de digitos del string. El patron `(\d+)` captura uno o mas digitos, y el `1` indica que queremos el primer grupo de captura. Luego `.cast("int")` convierte a entero. Las peliculas tipicamente se distribuyen alrededor de 90-100 minutos con una cola hacia la derecha (peliculas mas largas). La mayoria de series en Netflix tienen solo 1 temporada. La linea vertical con `axvline()` marca la media en el histograma, lo cual ayuda a interpretar la distribucion.

---
## Resumen

En esta actividad aprendimos:

1. **Exploracion de datos** con conteo de nulls por columna
2. **Distribucion de categorias** con groupBy y graficos de barras
3. **Split + Explode** para descomponer columnas con valores multiples
4. **Parseo de fechas** con formatos personalizados
5. **Extraccion de texto** con regexp_extract
6. **Visualizaciones** variadas: barras, lineas, histogramas

In [ ]:
# Detener SparkSession
spark.stop()
print("SparkSession detenida correctamente.")